In [1]:
import os
from dotenv import load_dotenv
from langchain_cerebras import ChatCerebras
from langchain_community.graphs import Neo4jGraph
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from neo4j import GraphDatabase
import httpx 
import time

/workspaces/pi-bench/.venv/lib/python3.12/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
load_dotenv()
AGENTBEATS = os.getenv("AGENT_BEATS")
if not AGENTBEATS:
    raise ValueError("Api key not loaded")

In [3]:
#neo4j connections
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or 'neo4j'

kg = Neo4jGraph(
    url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE
)

In [4]:

def connect_to_neo4j(retries=5, delay=3):
    for attempt in range(retries):
        try:
            kg = Neo4jGraph(
                url=NEO4J_URI,
                username=NEO4J_USERNAME,
                password=NEO4J_PASSWORD,
                database=NEO4J_DATABASE
            )
            # Test the connection
            kg.query("RETURN 1")
            print("✅ Connected to Neo4j")
            return kg
        except Exception as e:
            print(f"Attempt {attempt + 1} failed. Retrying in {delay}s...")
            time.sleep(delay)
    raise Exception("Could not connect to Neo4j after multiple attempts")


def wake_neo4j():
    print("Waking up Neo4j instance...")
    time.sleep(5)  # give it time to resume
    
wake_neo4j()
kg = connect_to_neo4j()

Waking up Neo4j instance...


✅ Connected to Neo4j


In [5]:
# Test Neo4j connection
result = kg.query("MATCH (n) RETURN labels(n), count(n) as count ORDER BY count DESC")
print(result)

[{'labels(n)': ['RedFlag'], 'count': 97}, {'labels(n)': ['RedFlagCategory'], 'count': 6}, {'labels(n)': ['Organization'], 'count': 5}, {'labels(n)': ['Obligation'], 'count': 5}, {'labels(n)': ['Rule'], 'count': 4}, {'labels(n)': ['RegulatoryNotice'], 'count': 1}]


In [6]:
kg.refresh_schema()
print(kg.schema)

Node properties:
RegulatoryNotice {id: STRING, title: STRING, topic: STRING, date: STRING, issuer: STRING, summary: STRING}
Rule {id: STRING, name: STRING, fullName: STRING, requirement: STRING, citation: STRING, threshold: STRING, description: STRING}
Obligation {id: STRING, name: STRING, threshold: STRING, description: STRING, deadline: STRING, reviewPeriod: STRING, filingDeadline: STRING, thresholdNote: STRING, citation: STRING, period: STRING, triggers: STRING, contact: STRING, requirement: STRING}
RedFlagCategory {id: STRING, name: STRING, sectionNumber: STRING, flagCount: INTEGER, note: STRING}
RedFlag {id: STRING, text: STRING, keywords: LIST, threshold: STRING, thresholdNote: STRING}
Organization {id: STRING, name: STRING, fullName: STRING, hotline: STRING}
Relationship properties:

The relationships:
(:RegulatoryNotice)-[:REFERENCES]->(:Rule)
(:RegulatoryNotice)-[:SUPERSEDES]->(:Rule)
(:RegulatoryNotice)-[:CONTAINS_CATEGORY]->(:RedFlagCategory)
(:Rule)-[:REQUIRES]->(:Obligatio

In [7]:
llm = ChatCerebras(
    model="llama3.1-8b",
    temperature=0.7,
    api_key=AGENTBEATS)



In [8]:
cypher_chain = GraphCypherQAChain.from_llm(
    llm, graph=kg, verbose=True, allow_dangerous_requests=True
)


In [9]:
def query_neo4j(question: str) -> str:
    """Queries the Neo4j graph database to answer questions about the data schema and entities."""
    return cypher_chain.run(question)

In [12]:
# Test 1: Simple count (should work)
print(query_neo4j("How many nodes are there?"))

# Test 2: Node labels only
print(query_neo4j("What are the node labels in the database?"))

# Test 3: Relationship count only
print(query_neo4j("How many relationships are there?"))



> Entering new GraphCypherQAChain chain...


Generated Cypher:
MATCH (n) RETURN count(n)
Full Context:
[{'count(n)': 118}]

> Finished chain.
There are 118 nodes.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (n) RETURN TYPE(n)


CypherSyntaxError: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Type mismatch: expected Relationship but was Node (line 1, column 23 (offset: 22))
"MATCH (n) RETURN TYPE(n)"
                       ^} {gql_status: 22G03} {gql_status_description: error: data exception - invalid value type}